# 02 — Run Comparison

**Purpose:** Manually explore, compare, and rank MLflow runs within an experiment using the Databricks SDK — mirroring the logic behind the `GET /runs/compare` (DS-006) and `GET /runs/{run_id}/compare-deployed` (DS-007) endpoints.

**What you'll learn:**
- How to fetch all runs for an experiment via `client.experiments.search_runs()`
- How to extract metrics and params from the SDK response
- How to implement a winner-selection heuristic (accuracy + RMSE)
- How to compare a candidate run against the deployed model from the Model Registry
- How to visualise metric comparisons with `matplotlib`

**Prerequisites:**
- `databricks-sdk`, `matplotlib`, `pandas` installed
- `.env` file with `DATABRICKS_HOST` and `DATABRICKS_TOKEN` (or mock mode)

---

## 1. Setup & Authentication

In [ ]:
import os
import json
from pathlib import Path
from datetime import datetime, timezone
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path('..') / 'backend' / '.env')

DATABRICKS_HOST  = os.getenv('DATABRICKS_HOST')
DATABRICKS_TOKEN = os.getenv('DATABRICKS_TOKEN')
USE_MOCK = not (DATABRICKS_HOST and DATABRICKS_TOKEN)

if not USE_MOCK:
    from databricks.sdk import WorkspaceClient
    client = WorkspaceClient(host=DATABRICKS_HOST, token=DATABRICKS_TOKEN)
    print(f"Connected to: {DATABRICKS_HOST}")
else:
    client = None
    print("MOCK mode — using embedded sample data")

## 2. Mock Run Data

These mirror the mock data in `services/run_service.py` so the notebook behaviour matches the API.

In [ ]:
MOCK_RUNS = [
    {
        "run_id": "abc123xyz",
        "experiment_id": "1",
        "status": "FINISHED",
        "start_time": "2026-03-28T10:05:00Z",
        "end_time":   "2026-03-28T10:45:00Z",
        "params":  {"n_estimators": "200", "max_depth": "6",  "learning_rate": "0.1"},
        "metrics": {"accuracy": 0.91, "f1_score": 0.88, "rmse": 0.12, "auc_roc": 0.94},
        "tags":    {"mlflow.user": "ds@astrazeneca.com"},
    },
    {
        "run_id": "def456uvw",
        "experiment_id": "1",
        "status": "FINISHED",
        "start_time": "2026-03-29T09:00:00Z",
        "end_time":   "2026-03-29T09:55:00Z",
        "params":  {"n_estimators": "300", "max_depth": "8",  "learning_rate": "0.05"},
        "metrics": {"accuracy": 0.93, "f1_score": 0.91, "rmse": 0.09, "auc_roc": 0.96},
        "tags":    {"mlflow.user": "ds2@astrazeneca.com"},
    },
]

MOCK_DEPLOYED = {
    "model_name": "drug_efficacy_prod",
    "version": "3",
    "metrics": {"accuracy": 0.89, "f1_score": 0.86, "rmse": 0.14, "auc_roc": 0.92},
}

print(f"{len(MOCK_RUNS)} mock runs loaded")

## 3. Fetch Runs for an Experiment

Use `client.experiments.search_runs()` with the target experiment ID.

In [ ]:
TARGET_EXPERIMENT_ID = "1"  # change as needed

if client is not None:
    raw_runs = list(client.experiments.search_runs(
        experiment_ids=[TARGET_EXPERIMENT_ID],
        max_results=100,
    ))
    runs = []
    for r in raw_runs:
        info = r.info
        data = r.data
        runs.append({
            "run_id":        info.run_id,
            "experiment_id": info.experiment_id,
            "status":        info.status.value if info.status else "UNKNOWN",
            "params":  {p.key: p.value for p in (data.params  or [])},
            "metrics": {m.key: m.value for m in (data.metrics or [])},
            "tags":    {t.key: t.value for t in (data.tags    or [])},
        })
else:
    runs = [r for r in MOCK_RUNS if r["experiment_id"] == TARGET_EXPERIMENT_ID]

print(f"Fetched {len(runs)} run(s) for experiment '{TARGET_EXPERIMENT_ID}'\n")
for r in runs:
    print(f"  run_id={r['run_id']}  status={r['status']}")
    print(f"    metrics: {r['metrics']}")
    print(f"    params:  {r['params']}")
    print()

## 4. Compare Runs Side by Side

Implements the same logic as `services/run_service.py::compare_runs()` and the `GET /runs/compare` endpoint (DS-006).

In [ ]:
import pandas as pd

# Build a comparison DataFrame
records = []
for r in runs:
    row = {"run_id": r["run_id"], "status": r["status"]}
    row.update({f"metric:{k}": v for k, v in r["metrics"].items()})
    row.update({f"param:{k}": v  for k, v in r["params"].items()})
    records.append(row)

df = pd.DataFrame(records).set_index("run_id")
print("Run Comparison Table:")
df

## 5. Determine the Winner

Uses the same heuristic as the backend: highest accuracy wins; ties broken by lowest RMSE.

In [ ]:
def determine_winner(runs_list):
    """Return (winner_run_id, reason) using accuracy + RMSE heuristic."""
    scored = []
    for r in runs_list:
        acc  = r["metrics"].get("accuracy", 0.0)
        rmse = r["metrics"].get("rmse", float("inf"))
        scored.append((r["run_id"], acc, rmse))

    scored.sort(key=lambda x: (-x[1], x[2]))  # desc accuracy, asc rmse
    best_id, best_acc, best_rmse     = scored[0]
    _,       second_acc, second_rmse = scored[1] if len(scored) > 1 else (None, 0, 0)

    if best_acc > second_acc and best_rmse < second_rmse:
        reason = "Higher accuracy and lower RMSE"
    elif best_acc > second_acc:
        reason = "Higher accuracy"
    elif best_rmse < second_rmse:
        reason = "Lower RMSE"
    else:
        reason = "Best overall metric profile"

    return best_id, reason


if len(runs) >= 2:
    winner_id, reason = determine_winner(runs)
    print(f"Winner: {winner_id}")
    print(f"Reason: {reason}")
else:
    print("Need at least 2 runs to compare")

## 6. Compare Candidate vs Deployed Model

Mirrors `services/run_service.py::compare_run_vs_deployed()` — DS-007 logic.

In [ ]:
CANDIDATE_RUN_ID  = "def456uvw"  # change to the run you want to evaluate
DEPLOYED_MODEL_NAME = "drug_efficacy_prod"

# Fetch candidate metrics
candidate = next((r for r in runs if r["run_id"] == CANDIDATE_RUN_ID), None)
if candidate is None and not USE_MOCK:
    # Fetch from Databricks directly
    raw = client.experiments.get_run(run_id=CANDIDATE_RUN_ID)
    d = raw.run.data
    candidate = {
        "run_id": CANDIDATE_RUN_ID,
        "metrics": {m.key: m.value for m in (d.metrics or [])},
        "params":  {p.key: p.value for p in (d.params  or [])},
    }

# Fetch deployed model metrics
if client is not None:
    try:
        versions = list(client.model_registry.get_latest_versions(
            name=DEPLOYED_MODEL_NAME, stages=["Production"]
        ))
        v = versions[0]
        dep_run = client.experiments.get_run(run_id=v.run_id)
        dep_d = dep_run.run.data
        deployed = {
            "model_name": DEPLOYED_MODEL_NAME,
            "version": str(v.version),
            "metrics": {m.key: m.value for m in (dep_d.metrics or [])},
        }
    except Exception as e:
        print(f"Could not fetch deployed model: {e} — using mock")
        deployed = MOCK_DEPLOYED
else:
    deployed = MOCK_DEPLOYED

# Comparison
cand_acc = candidate["metrics"].get("accuracy", 0)
dep_acc  = deployed["metrics"].get("accuracy", 0)
delta    = round(cand_acc - dep_acc, 4)

if delta > 0.01:
    recommendation = "PROMOTE"
    reason = "Candidate outperforms on all metrics"
elif delta < -0.01:
    recommendation = "REJECT"
    reason = "Candidate underperforms the deployed model"
else:
    recommendation = "HOLD"
    reason = "Marginal difference — further evaluation recommended"

print("=== Candidate vs Deployed ===")
print(f"Candidate run:    {candidate['run_id']}  accuracy={cand_acc}")
print(f"Deployed model:   {deployed['model_name']} v{deployed['version']}  accuracy={dep_acc}")
print(f"Accuracy delta:   {delta:+.4f}")
print(f"Recommendation:   {recommendation}")
print(f"Reason:           {reason}")

## 7. Metric Visualisation

Bar chart comparing key metrics across all runs plus the deployed baseline.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

METRICS_TO_PLOT = ["accuracy", "f1_score", "auc_roc"]

labels     = [r["run_id"][:10] + "..." for r in runs] + [f"deployed v{deployed['version']}"]
all_runs   = runs + [deployed]

x          = np.arange(len(METRICS_TO_PLOT))
bar_width  = 0.8 / len(all_runs)

fig, ax = plt.subplots(figsize=(10, 5))

for i, run in enumerate(all_runs):
    values = [run["metrics"].get(m, 0) for m in METRICS_TO_PLOT]
    offset = (i - len(all_runs) / 2 + 0.5) * bar_width
    bars = ax.bar(x + offset, values, bar_width, label=labels[i])
    ax.bar_label(bars, fmt="%.3f", padding=2, fontsize=7)

ax.set_xlabel("Metric")
ax.set_ylabel("Score")
ax.set_title("Run Comparison — Key Metrics")
ax.set_xticks(x)
ax.set_xticklabels(METRICS_TO_PLOT)
ax.set_ylim(0.75, 1.02)
ax.legend(loc="lower right")
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()
print(f"Winner: {winner_id if len(runs) >= 2 else 'N/A'} ({reason if len(runs) >= 2 else ''})")

## 8. Summary

This notebook validated:
- `GET /experiments/{id}/runs` (DS-003) — fetching and listing runs via SDK
- `GET /runs/compare` (DS-006) — winner determination logic
- `GET /runs/{run_id}/compare-deployed` (DS-007) — candidate vs production model

**Next:** See `03_schema_validation.ipynb` to verify the PostgreSQL schema and database connections.